<a href="https://colab.research.google.com/github/nusratjannat-2001/AI-lab/blob/main/parralel_practice_lab_exam_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile kernel.cu
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void matrixMulKernel(float *A, float *B, float *C, int M, int N, int P, int offset) {
    int k = threadIdx.x + offset;

    float *a = A + k * M * N;
    float *b = B + k * N * P;
    float *c = C + k * M * P;

    for (int i = 0; i < M; i++) {
        for (int j = 0; j < P; j++) {
            float sum = 0.0f;
            for (int l = 0; l < N; l++) {
                sum += a[i * N + l] * b[l * P + j];
            }
            c[i * P + j] = sum;
        }
    }
}

int main(int argc, char *argv[]) {
    if (argc < 3) {
        cout << "Usage: " << argv[0] << " <num_matrices> <max_threads>" << endl;
        return 1;
    }

    int K = stoi(argv[1]);  // number of matrices
    int x = stoi(argv[2]);  // number of threads

    int M = 400, N = 400, P = 400;
    int sizeA = K * M * N * sizeof(float);
    int sizeB = K * N * P * sizeof(float);
    int sizeC = K * M * P * sizeof(float);

    float *h_A = new float[K * M * N];
    float *h_B = new float[K * N * P];
    float *h_C = new float[K * M * P];

    // random initialization
    for (int i = 0; i < K * M * N; i++) h_A[i] = rand();
    for (int i = 0; i < K * N * P; i++) h_B[i] = rand();

    //copy data to device
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B, sizeB);
    cudaMalloc(&d_C, sizeC);

    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);

    // launch kernel
    // We have to process batch by batch
    // as we may not have enough threads to process all matrices at once
    for (int i = 0; i < K; i += x) {
        int currentBatchSize = min(x, K - i);
        printf("Processing matrixs from %d to %d\n", i, i + currentBatchSize - 1);
        matrixMulKernel<<<1, currentBatchSize>>>(d_A, d_B, d_C, M, N, P, i);
        cudaDeviceSynchronize();
    }

    //sync & copy
    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);

    // Output results
    /*
    for (int k = 0; k < K; k++) {
        cout << "Matrix C[" << k << "]: " << endl;
        for (int i = 0; i < M; i++) {
            for (int j = 0; j < P; j++) {
                cout << h_C[k * M * P + i * P + j] << " ";
            }
            cout << endl;
        }
        cout << endl;
    }
    */
    cout << "All multiplications completed successfully!" << endl;

    delete[] h_A;
    delete[] h_B;
    delete[] h_C;
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

}

Writing kernel.cu


In [ ]:
!nvcc -arch=sm_75 kernel.cu -o kernel

In [ ]:
!time ./kernel 10 1 && sleep 2

Processing matrixs from 0 to 0
Processing matrixs from 1 to 1
Processing matrixs from 2 to 2
Processing matrixs from 3 to 3
Processing matrixs from 4 to 4
Processing matrixs from 5 to 5
Processing matrixs from 6 to 6
Processing matrixs from 7 to 7
Processing matrixs from 8 to 8
Processing matrixs from 9 to 9
All multiplications completed successfully!

real	0m11.577s
user	0m11.158s
sys	0m0.256s


In [ ]:
!time ./kernel 10 2 && sleep 2

Processing matrixs from 0 to 1
Processing matrixs from 2 to 3
Processing matrixs from 4 to 5
Processing matrixs from 6 to 7
Processing matrixs from 8 to 9
All multiplications completed successfully!

real	0m6.133s
user	0m5.884s
sys	0m0.227s


In [ ]:
!time ./kernel 10 3 && sleep 2

Processing matrixs from 0 to 2
Processing matrixs from 3 to 5
Processing matrixs from 6 to 8
Processing matrixs from 9 to 9
All multiplications completed successfully!

real	0m4.978s
user	0m4.703s
sys	0m0.215s


In [ ]:
!time ./kernel 10 4 && sleep 2

Processing matrixs from 0 to 3
Processing matrixs from 4 to 7
Processing matrixs from 8 to 9
All multiplications completed successfully!

real	0m3.874s
user	0m3.551s
sys	0m0.214s


In [ ]:
!time ./kernel 10 10 && sleep 2

Processing matrixs from 0 to 9
All multiplications completed successfully!

real	0m1.957s
user	0m1.707s
sys	0m0.217s


In [ ]:
!time ./kernel 10 15 && sleep 2

Processing matrixs from 0 to 9
All multiplications completed successfully!

real	0m1.963s
user	0m1.740s
sys	0m0.216s


In [ ]:
!time ./kernel 10 20 && sleep 2

Processing matrixs from 0 to 9
All multiplications completed successfully!

real	0m1.962s
user	0m1.739s
sys	0m0.216s


In [ ]:
# %%writefile phonebook_search.cu
# #include <bits/stdc++.h>
# using namespace std;
# #include <cuda.h>

# using namespace std;

# struct Contact
# {
#     char name[256];
#     char phone_number[256];
# };

# __device__ bool check(char* name, char *search_name, int length)
# {
#     for (int i = 0; name[i] != '\0'; i++)
#     {
#         int j = 0;
#         while (name[i + j] != '\0' && search_name[j] != '\0' && name[i + j] == search_name[j])
#         {
#             j++;
#         }
#         if (j == length - 1)
#         {
#             return true;
#         }
#     }
#     return false;
# }

# __global__ void searchPhonebook(Contact* phonebook, int size, char* search_name, int length)
# {
#     int index = blockIdx.x * blockDim.x + threadIdx.x;

#     if (index < size)
#     {
#         if (check(phonebook[index].name, search_name, length))
#         {
#             printf("Name: %s, Phone: %s\n", phonebook[index].name, phonebook[index].phone_number);
#         }
#     }
# }

# int main(int argc, char* argv[])
# {
#     if (argc != 3) {
#         cerr << "Usage: " << argv[0] << " <search_name> <num_threads>" << endl;
#         return 1;
#     }

#     string search_name = argv[1];
#     int num_threads = atoi(argv[2]);
#     vector<string> file_names = {"phonebook1.txt", "phonebook2.txt"};
#     vector<Contact> phonebook;

#     for (auto file_name: file_names)
#     {
#         ifstream file(file_name);
#         Contact contact;
#         while (file >> contact.name >> contact.phone_number)
#         {
#             phonebook.push_back(contact);
#         }
#         file.close();
#     }
#     int size = phonebook.size();
#     Contact* device_phonebook;
#     cudaMalloc((void **)&device_phonebook, sizeof(Contact) * size);

#     cudaMemcpy(device_phonebook, phonebook.data(), sizeof(Contact) * size, cudaMemcpyHostToDevice);

#     int name_length = search_name.size() + 1;
#     char* device_search_name;
#     cudaMalloc((void **)&device_search_name, name_length);

#     cudaMemcpy(device_search_name, search_name.c_str(), name_length, cudaMemcpyHostToDevice);

#     for (int i = 0; i < size; i += num_threads) {
#         int currentBatchSize = min(num_threads, size - i);
#         searchPhonebook<<<1, currentBatchSize>>>(device_phonebook + i, currentBatchSize, device_search_name, name_length);
#         cudaDeviceSynchronize();

#         cudaError_t err = cudaGetLastError();
#         if (err != cudaSuccess) {
#             cout << "CUDA error: " << cudaGetErrorString(err) << endl;
#         }

#     }

#     cudaFree(device_phonebook);
#     cudaFree(device_search_name);
#     return 0;
# }
# %%writefile phonebook_search.cu
# #include <bits/stdc++.h>
# using namespace std;
# #include <cuda.h>

# struct Contact {
#     char name[256];
#     char phone_number[256];
# };

# __device__ bool check(char* name, char* search_name, int length) {
#     for (int i = 0; name[i] != '\0'; i++) {
#         int j = 0;
#         while (name[i + j] != '\0' && search_name[j] != '\0' && name[i + j] == search_name[j]) {
#             j++;
#         }
#         if (j == length - 1) {
#             return true;
#         }
#     }
#     return false;
# }

# __global__ void searchPhonebook(Contact* phonebook, int size, char* search_name, int length) {
#     int index = blockIdx.x * blockDim.x + threadIdx.x;

#     if (index < size) {
#         if (check(phonebook[index].name, search_name, length)) {
#             printf("Name: %s, Phone: %s\n", phonebook[index].name, phonebook[index].phone_number);
#         }
#     }
# }

# int main(int argc, char* argv[]) {
#     if (argc != 3) {
#         cerr << "Usage: " << argv[0] << " <search_name> <num_threads>" << endl;
#         return 1;
#     }

#     string search_name = argv[1];
#     int num_threads = atoi(argv[2]);
#     vector<string> file_names = {"phonebook1.txt", "phonebook2.txt"}; // You can update this list as needed
#     vector<Contact> phonebook;

#     for (const auto& file_name : file_names) {
#         ifstream file(file_name);
#         string line;
#         while (getline(file, line)) {
#             Contact contact;
#             size_t first_quote = line.find('"');
#             size_t second_quote = line.find('"', first_quote + 1);
#             size_t third_quote = line.find('"', second_quote + 1);
#             size_t fourth_quote = line.find('"', third_quote + 1);

#             if (first_quote != string::npos && second_quote != string::npos &&
#                 third_quote != string::npos && fourth_quote != string::npos) {

#                 string name = line.substr(first_quote + 1, second_quote - first_quote - 1);
#                 string phone = line.substr(third_quote + 1, fourth_quote - third_quote - 1);

#                 strncpy(contact.name, name.c_str(), sizeof(contact.name));
#                 strncpy(contact.phone_number, phone.c_str(), sizeof(contact.phone_number));
#                 contact.name[sizeof(contact.name) - 1] = '\0';
#                 contact.phone_number[sizeof(contact.phone_number) - 1] = '\0';

#                 phonebook.push_back(contact);
#             }
#         }
#         file.close();
#     }

#     int size = phonebook.size();
#     Contact* device_phonebook;
#     cudaMalloc((void**)&device_phonebook, sizeof(Contact) * size);
#     cudaMemcpy(device_phonebook, phonebook.data(), sizeof(Contact) * size, cudaMemcpyHostToDevice);

#     int name_length = search_name.size() + 1;
#     char* device_search_name;
#     cudaMalloc((void**)&device_search_name, name_length);
#     cudaMemcpy(device_search_name, search_name.c_str(), name_length, cudaMemcpyHostToDevice);

#     for (int i = 0; i < size; i += num_threads) {
#         int currentBatchSize = min(num_threads, size - i);
#         searchPhonebook<<<1, currentBatchSize>>>(device_phonebook + i, currentBatchSize, device_search_name, name_length);
#         cudaDeviceSynchronize();

#         cudaError_t err = cudaGetLastError();
#         if (err != cudaSuccess) {
#             cerr << "CUDA error: " << cudaGetErrorString(err) << endl;
#         }
#     }

#     cudaFree(device_phonebook);
#     cudaFree(device_search_name);
#     return 0;
# }


# %%writefile phonebook_search.cu
# #include <bits/stdc++.h>
# using namespace std;
# #include <cuda.h>

# struct Contact {
#     char name[256];
#     char phone_number[256];
# };

# __device__ bool match(char* name, char* search_name) {
#     int i = 0;
#     while (name[i] != '\0' && search_name[i] != '\0') {
#         if (name[i] != search_name[i]) return false;
#         i++;
#     }
#     return name[i] == '\0' && search_name[i] == '\0';
# }

# __global__ void searchPhonebook(Contact* phonebook, int phonebook_size,
#                                 char* search_names, int* name_offsets, int search_count) {
#     int index = blockIdx.x * blockDim.x + threadIdx.x;

#     if (index < phonebook_size) {
#         for (int i = 0; i < search_count; ++i) {
#             char* search_name = search_names + name_offsets[i];
#             if (match(phonebook[index].name, search_name)) {
#                 printf("Name: %s, Phone: %s\n", phonebook[index].name, phonebook[index].phone_number);
#                 break;
#             }
#         }
#     }
# }

# int main(int argc, char* argv[]) {
#     if (argc < 3) {
#         cerr << "Usage: " << argv[0] << " <name1> <name2> ... <num_threads>" << endl;
#         return 1;
#     }

#     int num_threads = atoi(argv[argc - 1]);
#     int search_count = argc - 2;

#     // Read phonebook
#     vector<string> file_names = {"phonebook1.txt", "phonebook2.txt"};
#     vector<Contact> phonebook;

#     for (const auto& file_name : file_names) {
#         ifstream file(file_name);
#         string line;
#         while (getline(file, line)) {
#             Contact contact;
#             size_t q1 = line.find('"'), q2 = line.find('"', q1 + 1);
#             size_t q3 = line.find('"', q2 + 1), q4 = line.find('"', q3 + 1);

#             if (q1 != string::npos && q2 != string::npos && q3 != string::npos && q4 != string::npos) {
#                 string name = line.substr(q1 + 1, q2 - q1 - 1);
#                 string phone = line.substr(q3 + 1, q4 - q3 - 1);

#                 strncpy(contact.name, name.c_str(), sizeof(contact.name));
#                 strncpy(contact.phone_number, phone.c_str(), sizeof(contact.phone_number));
#                 phonebook.push_back(contact);
#             }
#         }
#         file.close();
#     }

#     int phonebook_size = phonebook.size();

#     // Copy phonebook to device
#     Contact* d_phonebook;
#     cudaMalloc((void**)&d_phonebook, phonebook_size * sizeof(Contact));
#     cudaMemcpy(d_phonebook, phonebook.data(), phonebook_size * sizeof(Contact), cudaMemcpyHostToDevice);

#     // Flatten all names into a single array with offsets
#     vector<int> name_offsets(search_count);
#     int total_length = 0;
#     for (int i = 0; i < search_count; ++i) {
#         name_offsets[i] = total_length;
#         total_length += strlen(argv[i + 1]) + 1; // +1 for null terminator
#     }

#     char* all_names = new char[total_length];
#     for (int i = 0; i < search_count; ++i) {
#         strcpy(all_names + name_offsets[i], argv[i + 1]);
#     }

#     char* d_all_names;
#     int* d_offsets;
#     cudaMalloc((void**)&d_all_names, total_length * sizeof(char));
#     cudaMalloc((void**)&d_offsets, search_count * sizeof(int));
#     cudaMemcpy(d_all_names, all_names, total_length * sizeof(char), cudaMemcpyHostToDevice);
#     cudaMemcpy(d_offsets, name_offsets.data(), search_count * sizeof(int), cudaMemcpyHostToDevice);

#     // Launch kernel
#     for (int i = 0; i < phonebook_size; i += num_threads) {
#         int currentBatch = min(num_threads, phonebook_size - i);
#         searchPhonebook<<<1, currentBatch>>>(
#             d_phonebook + i, currentBatch, d_all_names, d_offsets, search_count
#         );
#         cudaDeviceSynchronize();

#         cudaError_t err = cudaGetLastError();
#         if (err != cudaSuccess) {
#             cerr << "CUDA Error: " << cudaGetErrorString(err) << endl;
#         }
#     }

#     cudaFree(d_phonebook);
#     cudaFree(d_all_names);
#     cudaFree(d_offsets);
#     delete[] all_names;

#     return 0;
# }

%%writefile phonebook_search.cu
#include <bits/stdc++.h>
using namespace std;
#include <cuda.h>

struct Contact {
    char name[256];
    char phone_number[256];
};

__device__ bool match(char* name, char* search_name) {
    int i = 0;
    while (name[i] != '\0' && search_name[i] != '\0') {
        if (name[i] != search_name[i]) return false;
        i++;
    }
    return name[i] == '\0' && search_name[i] == '\0';
}

__global__ void searchPhonebook(Contact* phonebook, int phonebook_size,
                                char* search_names, int* name_offsets, int search_count,
                                Contact* matched, int* match_count) {
    int index = blockIdx.x * blockDim.x + threadIdx.x;

    if (index < phonebook_size) {
        for (int i = 0; i < search_count; ++i) {
            char* search_name = search_names + name_offsets[i];
            if (match(phonebook[index].name, search_name)) {
                int pos = atomicAdd(match_count, 1);
                matched[pos] = phonebook[index];
                break;
            }
        }
    }
}

int main(int argc, char* argv[]) {
    if (argc < 3) {
        cerr << "Usage: " << argv[0] << " <name1> <name2> ... <num_threads>" << endl;
        return 1;
    }

    int num_threads = atoi(argv[argc - 1]);
    int search_count = argc - 2;

    // Load phonebook
    vector<string> file_names = {"phonebook1.txt", "phonebook2.txt"};
    vector<Contact> phonebook;

    for (const auto& file_name : file_names) {
        ifstream file(file_name);
        string line;
        while (getline(file, line)) {
            Contact contact;
            size_t q1 = line.find('"'), q2 = line.find('"', q1 + 1);
            size_t q3 = line.find('"', q2 + 1), q4 = line.find('"', q3 + 1);

            if (q1 != string::npos && q2 != string::npos && q3 != string::npos && q4 != string::npos) {
                string name = line.substr(q1 + 1, q2 - q1 - 1);
                string phone = line.substr(q3 + 1, q4 - q3 - 1);

                strncpy(contact.name, name.c_str(), sizeof(contact.name));
                strncpy(contact.phone_number, phone.c_str(), sizeof(contact.phone_number));
                phonebook.push_back(contact);
            }
        }
        file.close();
    }

    int phonebook_size = phonebook.size();

    // Prepare device memory
    Contact* d_phonebook;
    cudaMalloc((void**)&d_phonebook, phonebook_size * sizeof(Contact));
    cudaMemcpy(d_phonebook, phonebook.data(), phonebook_size * sizeof(Contact), cudaMemcpyHostToDevice);

    // Prepare search name memory
    vector<int> name_offsets(search_count);
    int total_length = 0;
    for (int i = 0; i < search_count; ++i) {
        name_offsets[i] = total_length;
        total_length += strlen(argv[i + 1]) + 1;
    }

    char* all_names = new char[total_length];
    for (int i = 0; i < search_count; ++i) {
        strcpy(all_names + name_offsets[i], argv[i + 1]);
    }

    char* d_all_names;
    int* d_offsets;
    cudaMalloc((void**)&d_all_names, total_length);
    cudaMalloc((void**)&d_offsets, search_count * sizeof(int));
    cudaMemcpy(d_all_names, all_names, total_length, cudaMemcpyHostToDevice);
    cudaMemcpy(d_offsets, name_offsets.data(), search_count * sizeof(int), cudaMemcpyHostToDevice);

    // Prepare matched result memory
    Contact* d_matched;
    int* d_match_count;
    int max_matches = phonebook_size;
    cudaMalloc((void**)&d_matched, max_matches * sizeof(Contact));
    cudaMalloc((void**)&d_match_count, sizeof(int));
    cudaMemset(d_match_count, 0, sizeof(int));

    // Launch kernel
    for (int i = 0; i < phonebook_size; i += num_threads) {
        int currentBatch = min(num_threads, phonebook_size - i);
        searchPhonebook<<<1, currentBatch>>>(
            d_phonebook + i, currentBatch,
            d_all_names, d_offsets, search_count,
            d_matched, d_match_count
        );
        cudaDeviceSynchronize();

        cudaError_t err = cudaGetLastError();
        if (err != cudaSuccess) {
            cerr << "CUDA Error: " << cudaGetErrorString(err) << endl;
        }
    }

    // Copy matched results back to host
    int h_match_count;
    cudaMemcpy(&h_match_count, d_match_count, sizeof(int), cudaMemcpyDeviceToHost);
    vector<Contact> matched(h_match_count);
    cudaMemcpy(matched.data(), d_matched, h_match_count * sizeof(Contact), cudaMemcpyDeviceToHost);

    // Write to file
    ofstream out("results.txt");
    for (const auto& contact : matched) {
        out << "Name: " << contact.name << ", Phone: " << contact.phone_number << "\n";
    }
    out.close();

    // Cleanup
    cudaFree(d_phonebook);
    cudaFree(d_all_names);
    cudaFree(d_offsets);
    cudaFree(d_matched);
    cudaFree(d_match_count);
    delete[] all_names;

    return 0;
}



Overwriting phonebook_search.cu


In [ ]:
!nvcc -arch=sm_75 phonebook_search.cu -o phonebook_search

In [ ]:
!time ./phonebook_search Sophie 1


real	0m0.212s
user	0m0.013s
sys	0m0.123s


In [ ]:
!time ./phonebook_search Sophie 2


real	0m0.126s
user	0m0.016s
sys	0m0.106s


In [ ]:
!time ./phonebook_search Sophie 3


real	0m0.132s
user	0m0.014s
sys	0m0.113s


In [ ]:
!cat phonebook1.txt

"FATEMA JAHAN TAMMY","015 05 040"
"SADIA BINTA M RAHMAN","017 62 031"
"TAHSINA HAQUE NABILA","014 58 886"
"SAZNIN AKTER ZITU","016 16 217"
"SAKIA RAHMAN","017 62 174"
"MASTURA ALAM","013 85 245"
"ASHAMONY","017 27 337"
"SAKIA RAHMAN","017 75 523"
"MASTURA ALAM","014 51 584"
"FARIA BINTE MOHASIN","017 57 844"
"SUMAIYA AKTER SWEETY","018 07 741"
"SUNJIDA AKTER NIPA","012 20 350"
"FATEMA JAHAN TAMMY","014 07 824"

In [ ]:
!time ./phonebook_search  "FATEMA JAHAN TAMMY" "MASTURA ALAM" 2


real	0m0.155s
user	0m0.015s
sys	0m0.128s


In [ ]:
!time ./phonebook_search Jack 1


real	0m0.128s
user	0m0.014s
sys	0m0.109s


In [ ]:
!time ./phonebook_search Jack 2

Name: Jack, Phone: 01333687273
Name: Jack, Phone: 01333975710

real	0m0.281s
user	0m0.021s
sys	0m0.242s


In [ ]:
!time ./phonebook_search Jack 3

Name: Jack, Phone: 01333687273
Name: Jack, Phone: 01333975710

real	0m0.230s
user	0m0.018s
sys	0m0.208s


In [ ]:
!time ./phonebook_search Jack 4

Name: Jack, Phone: 01333687273
Name: Jack, Phone: 01333975710

real	0m0.133s
user	0m0.018s
sys	0m0.111s


In [ ]:
!time ./phonebook_search sophie Jack 4

Usage: ./phonebook_search <search_name> <num_threads>

real	0m0.003s
user	0m0.001s
sys	0m0.001s


In [ ]:
%%writefile asif.cu
#include <iostream>
#include <cuda_runtime.h>
using namespace std;

__global__ void matrixMul(float* A, float* B, float* C, int M, int N, int P, int offset) {
    int k = threadIdx.x + offset;

    float* a = A + k * M * N;
    float* b = B + k * N * P;
    float* c = C + k * M * P;

    for(int i = 0; i < M; i++) {
        for(int j = 0; j < N; j++) {
            for(int l = 0; l < P; l++) {
                //c[i][l] += a[i][j] * b[j][l];
                c[i * P + l] = a[i * N + j] * b[j * P + l];
            }
        }
    }
}

int main(int argc, char *argv[]) {

    int T = atoi(argv[1]); //koyta thread use korte parbo
    int K = atoi(argv[2]); //koita matrix gun

    //100 gun, thread 10,

    int M = 400, N = 400, P = 400;

    int SizeA = M * N * K;
    int SizeB = N * P * K;
    int SizeC = M * P * K;

    //memory alocate (cpu allocate)
    float *h_A = new float[SizeA];
    float *h_B = new float[SizeB];
    float *h_C = new float[SizeC];


    //malloc (gpu allocate)
    float *d_A;
    cudaMalloc(&d_A, SizeA * sizeof(float));
    float *d_B;
    cudaMalloc(&d_B, SizeB * sizeof(float));
    float *d_C;
    cudaMalloc(&d_C, SizeC * sizeof(float));

    //data initialize
    for (int i = 0; i < SizeA; i++) {
        h_A[i] = rand();
    }
    for(int i = 0; i < SizeB; i++) {
        h_B[i] = rand();
    }


    //copy from host to device
    cudaMemcpy(d_A, h_A, SizeA * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, SizeB * sizeof(float), cudaMemcpyHostToDevice);

    //cuda process suru
    int gunKorteHobe = K;
    int offset = 0;
    while(gunKorteHobe > 0){

        int currentBatch = min(gunKorteHobe, T);

        matrixMul<<<1,currentBatch>>>(d_A, d_B, d_C, M, N, P, offset);
        cudaDeviceSynchronize();

        gunKorteHobe -= currentBatch;
        offset += currentBatch;
    }

    //let's copy back to cpu
    cudaMemcpy(h_C, d_C, SizeC * sizeof(float), cudaMemcpyDeviceToHost);

    cout << "All operation done" << endl;

}

Writing asif.cu


In [ ]:
!nvcc -arch=sm_75 asif.cu -o asif

In [ ]:
!time ./asif 1 10 && sleep 2

All operation done

real	0m30.145s
user	0m29.356s
sys	0m0.199s


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import pandas as pd


In [ ]:
data=pd.read_csv('/content/drive/MyDrive/labtest_dataset/labtest_dataset1.txt')

In [ ]:
data.head()

,FATEMA JAHAN TAMMY,015 05 040
0,SADIA BINTA M RAHMAN,017 62 031
1,TAHSINA HAQUE NABILA,014 58 886
2,SAZNIN AKTER ZITU,016 16 217
3,ANTU RANI HOWLADAR,017 62 174
4,MASTURA ALAM,013 85 245


In [ ]:
!nvidia-smi


Wed Apr 30 19:22:16 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile phonebook_search.cu
#include <bits/stdc++.h>
using namespace std;
#include <cuda.h>

struct Contact{
    char name[65];
    char phone_number[65];
};


string getInput(ifstream& file){
    string ans;
    char c;
    int readSuru = 0;
    while(file.get(c)){
        if(c == '\"'){
            if(readSuru == 1) break;
            readSuru = 1;
        }else{
            if(readSuru){
                ans.push_back(c);
            }
        }
    }
    return ans;
}

__device__ bool check(char* str1, char* str2){
    for(int i = 0; str1[i] != '\0'; i++){
        int flag = 1;
        for(int j = 0; str2[j] != '\0' ; j++){
            if(str1[i + j] != str2[j]){
                flag = 0;
                break;
            }
        }
        if(flag == 1) return true;
    }
    return false;
}


__global__ void myKernel(Contact* phoneBook, char* pat, int offset){
    int threadNumber = threadIdx.x + offset;
    if(check(phoneBook[threadNumber].name, pat)){
        printf("%s %s\n", phoneBook[threadNumber].name, phoneBook[threadNumber].phone_number);
    }
}



int main(int argc, char* argv[])
{
    int threadLimit = atoi(argv[2]);

    ifstream myfile("/content/drive/MyDrive/labtest_dataset/labtest_dataset1.txt");
    vector<Contact> phoneBook;

    int count = 0;

    while(myfile.peek() != EOF){

        if(count > 10000) break;
        count++;

        string name = getInput(myfile);
        string phoneNum = getInput(myfile);

        Contact c;
        strcpy(c.name, name.c_str());
        strcpy(c.phone_number, phoneNum.c_str());

        phoneBook.push_back(c);
    }

    string search_name = argv[1];
    char pat[65];
    strcpy(pat, search_name.c_str());


    char* d_pat;
    cudaMalloc(&d_pat, 65); //memory allocation
    cudaMemcpy(d_pat, pat, 65, cudaMemcpyHostToDevice); //copying to device

    int n = phoneBook.size();
    Contact* d_phoneBook;
    cudaMalloc(&d_phoneBook, n*sizeof(Contact));
    cudaMemcpy(d_phoneBook, phoneBook.data(), n * sizeof(Contact), cudaMemcpyHostToDevice);


    int bakiAche = n;
    int offset = 0;
    while(bakiAche > 0){
        int batchSize = min(threadLimit, bakiAche);
        myKernel<<<1,batchSize>>>(d_phoneBook, d_pat, offset);
        cudaDeviceSynchronize();

        bakiAche -= batchSize;
        offset += batchSize;
    }

}

Overwriting phonebook_search.cu


In [ ]:
!nvcc -arch=sm_75 phonebook_search.cu -o phonebook_search

In [ ]:
!time ./phonebook_search AKTER 5 > output.txt


real	0m0.249s
user	0m0.122s
sys	0m0.119s
